# Simulation of LUCJ circuits

In [ ]:
import ffsim
import matplotlib.pyplot as plt; plt.rcParams.update({"font.family": "serif", "font.size": 12})
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf

import qiskit
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import CouplingMap
from qiskit.quantum_info import Statevector, SparsePauliOp

## Parameters

In [ ]:
atom: str = "H"
natoms: int = 6
nlayers: int = 2  # Number of layers in the ansatz.

## Problem defintion

### Molecule

In [ ]:
def generate_linear_geometry(atom: str, natoms: int, atomic_distance: float = 1.0) -> str:
    """Returns a linear Hydrogen chain geometry for use in PySCF molecule construction.
    
    Args:
        natoms: Number of Hydrogen atoms in the chain.
        atomic_distance: Equal spacing between Hydrogen atoms.
    """
    return "; ".join([f"{atom} 0 0 {i * atomic_distance}" for i in range(natoms)])

In [ ]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=generate_linear_geometry(atom, natoms),
    basis="sto-6g",
)

# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
# reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

## Circuit construction

In [ ]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

In [ ]:
coupling_map = CouplingMap.from_grid(
    num_rows=int(np.ceil(np.sqrt(2 * norb))),
    num_columns=int(np.ceil(np.sqrt(2 * norb)))
)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

In [ ]:
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = [(p, p) for p in range(norb)]  # None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

In [ ]:
# Create pass manager
try:
    pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
        backend=backend,
        norb=norb,
        connectivity="square",
        interaction_pairs=(pairs_aa, pairs_ab),
        optimization_level=3,
    )
    print("Unable to generate ffsim pass manager")
except RuntimeError:
    pass_manager = None

print("pairs_aa:", pairs_aa)
print("pairs_ab:", pairs_ab)

In [ ]:
# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=nlayers,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    # options=dict(maxiter=1000),
)

qubits = qiskit.QuantumRegister(2 * norb, name="q")
circuit = qiskit.QuantumCircuit(qubits)
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)

In [ ]:
if pass_manager is not None:
    compiled = pass_manager.run(circuit)
else:
    compiled = qiskit.transpile(circuit, backend=backend, optimization_level=3)

In [ ]:
print(f"Number of qubits: {compiled.num_qubits}")
print(f"Gate counts: {compiled.count_ops()}")

In [ ]:
compiled.draw(fold=-1)

## Observable definition

In [ ]:
observables = [
    SparsePauliOp("ZZ" + "I" * (compiled.num_qubits - 2)),
    SparsePauliOp("I" * (compiled.num_qubits // 2) + "ZZ" + "I" * (compiled.num_qubits // 2 - 2)),
    SparsePauliOp("I" * (compiled.num_qubits - 2) + "ZZ")
]

## Exact values

In [ ]:
if compiled.num_qubits <= 20:
    expectation_values_exact = []
    for observable in observables:
        statevector = Statevector(compiled)
        sv_expectation_value = statevector.expectation_value(observable).real
        print(sv_expectation_value)
        expectation_values_exact.append(sv_expectation_value)

## Heisenberg simulation

In [ ]:
from propaq.datatypes.majorana import MajoranaMonomial

from propaq.propagators import MajoranaPropagator
from propaq.circuits import MajoranaCircuit 
from propaq.noise import UniformNoiseModel, truncation
from propaq.noise import TruncationPolicy 

from propaq.datatypes import MajoranaTermSum

In [ ]:
mc = MajoranaCircuit.from_qiskit(compiled.copy(), n_modes=2 * compiled.num_qubits)

In [ ]:
damping: float = 0.0000
cutoff: float = 1e-16
prop = MajoranaPropagator(
    UniformNoiseModel(damping=damping),
    TruncationPolicy(weight_cutoff=100000, coeff_cutoff=cutoff),
    n_threads=2,
    progress_bar=True,
)

In [ ]:
results = []
for observable in observables:
    observable_mts = MajoranaTermSum.from_sparse_pauli_op(observable)
    results.append(prop.expectation_value(observable_mts, mc, fock_state=0))
    print(results[-1].expectation_value)

In [ ]:
for i, result in enumerate(results):
    plt.semilogy(result.n_terms, "--o", alpha=0.55,label="ZZ " + ",".join(map(str, observables[i].to_sparse_list()[0][1])))

plt.xlabel("Gate index")
plt.ylabel("Number of Majorana monomials")
plt.title(f"{natoms} {atom} atom(s), {len(compiled.qubits)} qubits, {nlayers} LUCJ layer(s), noise damping = {damping}, cutoff = {cutoff}\n{compiled.count_ops()}")
plt.legend();

## Accuracy vs coefficient cutoff threshold

In [ ]:
cutoffs = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
evals = []
for cutoff in cutoffs:
    print("On cutoff:", cutoff)
    prop = MajoranaPropagator(
        UniformNoiseModel(damping=0.00),
        TruncationPolicy(weight_cutoff=100000, coeff_cutoff=cutoff),
        n_threads=2,
        progress_bar=True,
    )
    oevals = []
    for i, observable in enumerate(observables):
        observable_mts = MajoranaTermSum.from_sparse_pauli_op(observable)
        mp_expectation_value = prop.expectation_value(observable_mts, mc, fock_state=0).expectation_value
        print(mp_expectation_value, expectation_values_exact[i])
        oevals.append(mp_expectation_value)
    evals.append(oevals)

In [ ]:
evals = np.array(evals).T

errors = []
for i, e in enumerate(evals):
    errors.append(e - expectation_values_exact[i])

errors = np.abs(np.array(errors))

In [ ]:
for i, e in enumerate(errors):
    plt.loglog(cutoffs, e, "--s", ms=9, mec="black", alpha=0.75, label="ZZ " + ",".join(map(str, observables[i].to_sparse_list()[0][1])))

plt.xlabel("Coefficient magnitude cutoff")
plt.ylabel(f"Absolute error")
plt.legend()
plt.title(f"{natoms} {atom} atom(s), {len(compiled.qubits)} qubits, {nlayers} LUCJ layer(s)\n{compiled.count_ops()}");
plt.savefig("error_vs_cutoff.pdf")